In [1]:
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
# import geopandas as gpd
# from shapely.geometry import Point
import glob

from tqdm import tqdm
import os
import shutil
import tempfile
import logging
import geopandas as gpd
import rioxarray as rxr

In [29]:
BASE_FILE = Path.cwd () / "GloFASv5_stations_metadata_calfunction_KGE_JSD_20March2026_final.csv"
# Folder with static attributes
DIR_STATIC = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/static_maps/GloFASv5_staticmaps_consolidated_March2026/GloFASv5_static_maps_reanalysis/")
# Folder with parameter attributes
DIR_PARS = Path("/mnt/eos_rw/projects/FLOODS-RIVER/schafti/02_GloFAS_EFAS/GloFAS/GloFASv5/static_maps/GloFASv5_parametermaps/")

# 1. Check decimal separator Problem im BASE_FILE
df_base = pd.read_csv(BASE_FILE)

In [38]:
df_base[["ID", "name", "long", "lat", "LISFLOOD_X", "LISFLOOD_Y"]][df_base["ID"] == 6739]

,ID,name,long,lat,LISFLOOD_X,LISFLOOD_Y
2054,6739,Tuolba At Alekseevka,12400000.0,60.28,124.025,60.325


In [31]:
# Stationen mit unrealistischen Koordinaten
bad_coords = df_base[
    (df_base["long"].abs() > 180) |
    (df_base["lat"].abs() > 90)
]
print(f"Out of range: {len(bad_coords)}")
print(bad_coords[["ID", "name", "long", "lat", "LISFLOOD_X", "LISFLOOD_Y"]].head(20))

# 2. Check lat/lon flip wie gerade gefunden
df_base["diff_lon"] = (df_base["long"] - df_base["LISFLOOD_X"]).abs()
df_base["diff_lat"] = (df_base["lat"] - df_base["LISFLOOD_Y"]).abs()
suspicious = df_base[(df_base["diff_lon"] > 1) | (df_base["diff_lat"] > 1)]
print(f"Suspicious: {len(suspicious)}")
print(suspicious[["ID", "name", "long", "lat", "LISFLOOD_X", "LISFLOOD_Y"]])

merged = pd.concat([bad_coords, suspicious], ignore_index=True).drop_duplicates(subset="ID")
merged.to_csv(BASE_FILE.parent / "problematic_stations_coordinates.csv", index=False)

Out of range: 22
         ID                      name         long         lat  LISFLOOD_X  \
1263   5429                   Samburg        78.22  6700000.00      78.175   
2054   6739      Tuolba At Alekseevka  12400000.00       60.28     124.025   
3967  14573               TAZZOUGUERT        -3.75  3200000.00      -3.775   
4009  14740        KALUNGU (60773235)   3200000.00      -10.09      31.975   
4041  15037              YANGXINJIANG  10700000.00       33.05     106.875   
4045  15093                    DZEZDY   6700000.00       48.05      67.075   
4198  15418                     MAYMA        85.85  5200000.00      85.925   
4235  15551               EL PROFUNDO       -74.50   400000.00     -74.475   
4270  15625                ALGARROBAL       -70.59 -3000000.00     -70.575   
4291  15694        PASO DE LAS TOSCAS  -5500000.00      -32.13     -54.975   
4394  15900       NEAR PERRYVILLE. IL  -8900000.00       42.19     -89.025   
4488  16020        NEAR EDINBURGH. IN  -8600000

In [40]:
# ─────────────────────────────────────────────
# COORDINATE QC & CORRECTION
# ─────────────────────────────────────────────

df_base = pd.read_csv(BASE_FILE)

# ── Schritt 1: Decimal separator fix ──
bad_mask = (df_base["long"].abs() > 180) | (df_base["lat"].abs() > 90)
print(f"Fixing {bad_mask.sum()} stations with bad coordinates (decimal separator issue)")

# Dokumentation vor dem Fix
bad_coords = df_base.loc[bad_mask, ["ID", "name", "long", "lat", 
                                     "LISFLOOD_X", "LISFLOOD_Y"]].copy()
bad_coords["issue"] = "decimal_separator"

df_base.loc[bad_mask, "long"] = df_base.loc[bad_mask, "LISFLOOD_X"]
df_base.loc[bad_mask, "lat"]  = df_base.loc[bad_mask, "LISFLOOD_Y"]

# ── Schritt 2: Flip check nach decimal fix ──
df_base["diff_lon"] = (df_base["long"] - df_base["LISFLOOD_X"]).abs()
df_base["diff_lat"] = (df_base["lat"]  - df_base["LISFLOOD_Y"]).abs()

suspicious_after = df_base[
    (df_base["diff_lon"] > 1) | (df_base["diff_lat"] > 1)
].copy()
print(f"\nSuspicious after decimal fix: {len(suspicious_after)}")
print(suspicious_after[["ID", "name", "long", "lat", 
                         "LISFLOOD_X", "LISFLOOD_Y", "diff_lon", "diff_lat"]])

# Dokumentation — keine Korrektur, provider coords behalten
suspicious_after["issue"] = "coordinate_mismatch_unresolved"

# ── Schritt 3: Final check ──
bad_after = (df_base["long"].abs() > 180) | (df_base["lat"].abs() > 90)
print(f"\nRemaining out-of-range coordinates: {bad_after.sum()}")

# ── Schritt 4: Cleanup helper columns ──
df_base = df_base.drop(columns=["diff_lon", "diff_lat"])

# ── Schritt 5: Dokumentation speichern ──
doc_df = pd.concat([bad_coords, suspicious_after[["ID", "name", "long", "lat",
                                                    "LISFLOOD_X", "LISFLOOD_Y", 
                                                    "issue"]]], 
                    ignore_index=True).drop_duplicates(subset="ID")
doc_path = BASE_FILE.parent / "problematic_stations_coordinates.csv"
doc_df.to_csv(doc_path, index=False)
print(f"\nDocumentation saved to {doc_path}")

# ── Schritt 6: Korrigiertes File speichern ──
corrected_path = BASE_FILE.parent / BASE_FILE.name.replace(".csv", "_corrected.csv")
df_base.to_csv(corrected_path, index=False)
print(f"Corrected file saved to {corrected_path}")

Fixing 22 stations with bad coordinates (decimal separator issue)

Suspicious after decimal fix: 2
         ID              name   long    lat  LISFLOOD_X  LISFLOOD_Y  diff_lon  \
817    2275    AGAN NAHARAYIM  35.58  32.63      51.725      32.625    16.145   
4398  15905  CHARLES CITY. IA -92.67  43.06     -96.175      47.875     3.505   

      diff_lat  
817      0.005  
4398     4.815  

Remaining out-of-range coordinates: 0

Documentation saved to /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/problematic_stations_coordinates.csv
Corrected file saved to /storage/schafti/Documents/01_Hydrology/01_Lisflood/02_Data/03_GloFASv5/01_HydroBot/GloFASv5_stations_metadata_calfunction_KGE_JSD_20March2026_final_corrected.csv
